# 5. Matrix Chain Multiplication (MCM)

### Problem Statement

**The Challenge:**
You are given a sequence (chain) of matrices $A_1, A_2, \dots, A_n$. You need to calculate the product of these matrices:
$$A_1 \times A_2 \times \dots \times A_n$$

**Key Insight:**
Matrix multiplication is **Associative**: $(AB)C = A(BC)$.
However, the **number of scalar multiplications** (the computational cost) depends heavily on how we place the parentheses.

**Goal:**
Find the parenthesization order that leads to the **minimum number of multiplications**.

<center>
    <img src="./img/matrix.png">
</center>

---

### Understanding the Cost

If matrix $A$ has dimensions **$p \times q$** and matrix $B$ has dimensions **$q \times r$**:
1.  The resulting matrix has dimensions **$p \times r$**.
2.  The cost to compute this is **$p \times q \times r$** multiplications.

For example, using the matrices from the image above:
* **A**: $5 \times 5$
* **B**: $5 \times 4$

To compute **$(AB)$**:
* The cost is $5 \times 5 \times 4 = \mathbf{100}$ multiplications.
* The resulting matrix has dimensions $5 \times 4$.

---

### The Core Logic: *“Where do I split?”*

We cannot try every parenthesis combination brute-force (the number of ways grows exponentially). Instead, we use recursion.

For a chain of matrices from index $i$ to $j$:
We must split them at some index $k$ (between $i$ and $j-1$) into two parts:
1.  **Left Part:** $(A_i \dots A_k)$
2.  **Right Part:** $(A_{k+1} \dots A_j)$

The total cost for a specific split $k$ is:
$$
\text{Cost} = \underbrace{\text{MCM}(i, k)}_{\text{Cost of Left}} + \underbrace{\text{MCM}(k+1, j)}_{\text{Cost of Right}} + \underbrace{(p_{i-1} \times p_k \times p_j)}_{\text{Combine Cost}}
$$

We try **every possible $k$** and pick the minimum.

---

### Numerical Walkthrough

Let's calculate the cost for different parenthesizations of the product **$A \times B \times C \times D$** from the image.

**Matrix Dimensions:**
* $A$: $5 \times 5$
* $B$: $5 \times 4$
* $C$: $4 \times 8$
* $D$: $8 \times 2$

The final result will be a $5 \times 2$ matrix.

#### Option 1: Parenthesization $((AB)C)D$

1.  **Compute $(AB)$:**
    * Dimensions: $(5 \times 5) \times (5 \times 4) \rightarrow (5 \times 4)$
    * Cost: $5 \times 5 \times 4 = 100$
2.  **Compute $((AB)C)$:**
    * Dimensions: $(5 \times 4) \times (4 \times 8) \rightarrow (5 \times 8)$
    * Cost: $5 \times 4 \times 8 = 160$
3.  **Compute $(((AB)C)D)$:**
    * Dimensions: $(5 \times 8) \times (8 \times 2) \rightarrow (5 \times 2)$
    * Cost: $5 \times 8 \times 2 = 80$

* **Total Cost 1** $= 100 + 160 + 80 = \mathbf{340}$.

#### Option 2: Parenthesization $A(B(CD))$

1.  **Compute $(CD)$:**
    * Dimensions: $(4 \times 8) \times (8 \times 2) \rightarrow (4 \times 2)$
    * Cost: $4 \times 8 \times 2 = 64$
2.  **Compute $(B(CD))$:**
    * Dimensions: $(5 \times 4) \times (4 \times 2) \rightarrow (5 \times 2)$
    * Cost: $5 \times 4 \times 2 = 40$
3.  **Compute $(A(B(CD)))$:**
    * Dimensions: $(5 \times 5) \times (5 \times 2) \rightarrow (5 \times 2)$
    * Cost: $5 \times 5 \times 2 = 50$

* **Total Cost 2** $= 64 + 40 + 50 = \mathbf{154}$.

**Verdict:** Option 2 ($A(B(CD))$) is less than half the cost of Option 1 ($((AB)C)D$). This demonstrates the importance of finding the optimal order. The MCM algorithm finds the parenthesization with the absolute minimum cost.

---

### The Algorithm (Pseudocode)

```text
Algorithm MCM(p, i, j):

    1. Base Case:
       IF i == j:
           RETURN 0   (Single matrix costs 0 to multiply)

    2. Recursive Step:
       min_cost = Infinity

       # Try splitting at every index k from i to j-1
       FOR k FROM i TO j-1:
           
           # Calculate cost of this split
           current_cost = MCM(p, i, k)            # Solve Left side
                          + MCM(p, k+1, j)        # Solve Right side
                          + (p[i-1] * p[k] * p[j]) # Cost to multiply the two results

           # Keep the minimum
           IF current_cost < min_cost:
               min_cost = current_cost

    3. Return Result
       RETURN min_cost

```
### Complexity Analysis

Just like Knapsack and Subset Sum, this recursive approach re-calculates the same subproblems repeatedly.

* **Time Complexity:** Exponential $O(2^n)$.
* **Space Complexity:** $O(n)$ (Recursion stack).
* **Dynamic Programming Fix:** By using a 2D table `dp[i][j]` to store the results of subproblems, we reduce the time complexity to $O(n^3)$, which is the standard efficiency for solving MCM.


# 6. Optimal Binary Search Tree (OBST)

### Problem Statement

**The Challenge:**
You are given a sorted list of keys (e.g., $A, B, C, D$) and their associated **search probabilities** (how often each key is accessed).

**Goal:**
Construct a Binary Search Tree (BST) that minimizes the **Expected Search Cost**.
* **Search Cost:** The number of comparisons needed to find a key. This equals the **depth** of the node ($1$ for root, $2$ for children, etc.).
* **Strategy:** Place frequently accessed keys (high probability) closer to the Root.

---

### The Cost Function

The cost of a tree is calculated as:
$$
\text{Total Cost} = \sum (\text{Probability of Key} \times \text{Depth of Key})
$$

---

### Numerical Walkthrough

<center>
    <img src="./img/obst.png" width="700">
</center>


Let's look at the example provided in the image.

**Input Data:**
| Key | A | B | C | D |
| :--- | :---: | :---: | :---: | :---: |
| **Probability** | 0.1 | 0.2 | 0.4 | 0.3 |

#### Case 1: A Linear Tree (Skewed)
Imagine we simply insert them in order: $A \rightarrow B \rightarrow C \rightarrow D$.
* **Root A** (Level 1): $0.1 \times 1 = 0.1$
* **Child B** (Level 2): $0.2 \times 2 = 0.4$
* **Child C** (Level 3): $0.4 \times 3 = 1.2$
* **Child D** (Level 4): $0.3 \times 4 = 1.2$
* **Total Cost:** $0.1 + 0.4 + 1.2 + 1.2 = \mathbf{2.9}$

*This is inefficient because the most popular key (C) is deep in the tree.*

#### Case 2: The Optimal Tree
We want **C** (0.4) and **D** (0.3) near the top. The algorithm checks combinations and finds this structure:
* **Root:** C
* **Right Child:** D
* **Left Child:** B
    * **Left Child of B:** A

Let's calculate the cost:
* **Root C** (Level 1): $0.4 \times 1 = 0.4$
* **Child B** (Level 2): $0.2 \times 2 = 0.4$
* **Child D** (Level 2): $0.3 \times 2 = 0.6$
* **Child A** (Level 3): $0.1 \times 3 = 0.3$
* **Total Cost:** $0.4 + 0.4 + 0.6 + 0.3 = \mathbf{1.7}$

**Verdict:** The OBST algorithm reduces the average search cost from **2.9** to **1.7**.

---

### The Core Logic: *“Pick the Best Root”*

Just like Matrix Chain Multiplication, we solve this using **Dynamic Programming**.

For any range of keys from $i$ to $j$ (e.g., $A$ to $D$), we try making **every key** $r$ (from $i$ to $j$) the Root.

**If we pick key $r$ as the Root:**
1.  **Left Subtree:** Contains keys $i \dots r-1$.
2.  **Right Subtree:** Contains keys $r+1 \dots j$.
3.  **Cost Increase:** Because key $r$ is now the root (Level 1), *every* node in the Left and Right subtrees gets pushed down by 1 level. This adds the sum of all frequencies in the range to the cost.

**Recursive Formula:**
$$
\text{Cost}(i, j) = \min_{r=i \dots j} \left( \text{Cost}(i, r-1) + \text{Cost}(r+1, j) \right) + \text{SumOfFreq}(i, j)
$$

---


### The Algorithm (Recursive)

Just like Matrix Chain Multiplication, we check every possible root for a given range $(i, j)$ and pick the one that gives the minimum cost.

**Recursive Function `OBST(i, j)`:**
* **Input:** Range of keys from index $i$ to $j$.
* **Output:** Minimum cost to build a tree with these keys.

```text
Algorithm OBST_Recursive(freq, i, j):

    1. Base Case: No keys left
       IF i > j:
           RETURN 0

    2. Base Case: Single key
       IF i == j:
           RETURN freq[i]

    3. Recursive Step:
       min_cost = Infinity
       
       # Calculate sum of frequencies for this range (needed for cost offset)
       # Because every node goes 1 level deeper, we add the sum of all weights
       sum_weight = Sum(freq, i, j)

       # Try making every key 'r' the root (from i to j)
       FOR r FROM i TO j:
           
           # Cost = (Left Subtree) + (Right Subtree) + (Weight of current tree)
           current_cost = OBST_Recursive(freq, i, r-1) + 
                          OBST_Recursive(freq, r+1, j) + 
                          sum_weight

           # Keep the minimum
           IF current_cost < min_cost:
               min_cost = current_cost

    4. Return Result
       RETURN min_cost

```

### Complexity Analysis

* **Time Complexity:** $O(n^3)$
    * We have $O(n^2)$ subproblems (ranges $i$ to $j$).
    * For each subproblem, we loop through $O(n)$ choices for the root.
    * **Total:** $n \times n \times n$.

* **Space Complexity:** $O(n^2)$
    * To store the `Cost` table.

* **Note:** The complexity is identical to Matrix Chain Multiplication because the "Split Point" logic is structurally the same as the "Root Selection" logic.

# 7. Binomial Coefficient ($^nC_r$)

### Problem Statement

**The Challenge:**
Find the number of ways to choose $r$ items from a set of $n$ distinct items. This is denoted as $C(n, r)$ or $\binom{n}{r}$.

**Common Formula:**
$$C(n, r) = \frac{n!}{r!(n-r)!}$$

**The Computational Issue:**
Calculating factorials (like $100!$) results in massively large numbers that cause **Integer Overflow** in most programming languages. We need a more stable approach that doesn't calculate the full factorial.

---

### The Core Logic: *“Include or Exclude”*

We can solve this using the recursive property known as **Pascal's Identity**:
$$C(n, r) = \underbrace{C(n - 1, r - 1)}_{\text{Include Item}} + \underbrace{C(n - 1, r)}_{\text{Exclude Item}}$$

**The Logic:**
To choose $r$ items from $n$, we look at the last item:
1.  **Include it:** We take the item, so we need to choose $r-1$ more from the remaining $n-1$.
2.  **Exclude it:** We skip the item, so we still need to choose $r$ from the remaining $n-1$.

---

### Visual Walkthrough

Let's calculate **$C(5, 3)$** (Choosing 3 items from 5).

<center>
    <img src="./img/bionomial.png" width="400">
</center>

**Explanation of the Diagram:**
In the above diagram, our task is to simplify or use the recursion process until we reach the endpoint, which is 1. The values will become 1 by following the rule:
* **Base Case 1:** $C(n, 0) = 1$ (Choosing 0 items is 1 way: the empty set).
* **Base Case 2:** $C(n, n) = 1$ (Choosing all items is 1 way).

**Tracing the Tree:**
1.  **Start:** $C(5, 3)$ breaks down into $C(4, 2) + C(4, 3)$.
2.  **Step Down:** $C(4, 2)$ breaks down into $C(3, 1) + C(3, 2)$.
3.  **Bottom (Base Cases):**
    * $C(2, 0) = 1$
    * $C(2, 2) = 1$
    * $C(1, 0) = 1$
4.  **Summing Up:** The $1$s bubble up to form the final answer.
    * $C(5, 3) = 6 + 4 = \mathbf{10}$.

---

### The Algorithm (Recursive)

```text
Algorithm BinomialCoeff(n, r):

    1. Base Case:
       IF r == 0 OR r == n:
           RETURN 1

    2. Recursive Step:
       # Pascal's Identity
       # Sum of (Include current) + (Exclude current)
       RETURN BinomialCoeff(n - 1, r - 1) + BinomialCoeff(n - 1, r)

```

### Complexity Analysis

* **Time Complexity:** Exponential $O(2^n)$ [without optimization].
    * The recursion tree grows rapidly because subproblems like $C(3, 2)$ are recalculated multiple times.
* **Space Complexity:** $O(n)$ (Recursion stack depth).
* **Dynamic Programming Fix:**
    * By storing values in a table `Table[n][r]`, we can compute this in $O(n \times r)$ time.
    * This table construction is exactly how **Pascal's Triangle** is built!